<a href="https://colab.research.google.com/github/grasht/projects_ml_lab_3/blob/main/week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
%pip install evaluate

In [33]:
from transformers import pipeline
from datasets import load_dataset
import evaluate
import time
from PIL import Image

In [34]:
imdb_set = load_dataset("imdb")
squad_set = load_dataset("squad")

In [35]:
print(imdb_set.column_names)
print(squad_set.column_names)
print(squad_set.data)

{'train': ['text', 'label'], 'test': ['text', 'label'], 'unsupervised': ['text', 'label']}
{'train': ['id', 'title', 'context', 'question', 'answers'], 'validation': ['id', 'title', 'context', 'question', 'answers']}
{'train': MemoryMappedTable
id: string
title: string
context: string
question: string
answers: struct<text: list<item: string>, answer_start: list<item: int32>>
  child 0, text: list<item: string>
      child 0, item: string
  child 1, answer_start: list<item: int32>
      child 0, item: int32
----
id: [["5733be284776f41900661182","5733be284776f4190066117f","5733be284776f41900661180","5733be284776f41900661181","5733be284776f4190066117e",...,"56d4f71e2ccc5a1400d833aa","56bed4553aeaaa14008c94e5","56bed4553aeaaa14008c94e7","56bed4553aeaaa14008c94e8","56bfe7eaa10cfb1400551387"],["56bfe7eaa10cfb1400551389","56bfe7eaa10cfb140055138a","56d4f7a22ccc5a1400d833ae","56d4f7a22ccc5a1400d833b0","56d4f7a22ccc5a1400d833b1",...,"56d381b459d6e41400146594","56d381b459d6e41400146595","56cfe91

# Sentiment Analysis

In [36]:
small_dataset = imdb_set["test"].shuffle(seed=7).select(range(100))

analyzer = pipeline("sentiment-analysis", device=0)

predictions = []
references = []

for example in small_dataset:
    result = analyzer(example["text"][:512])[0]  # truncate long text
    label = 1 if result["label"] == "POSITIVE" else 0

    predictions.append(label)
    references.append(example["label"])

accuracy = evaluate.load("accuracy")

results = accuracy.compute(predictions=predictions, references=references)

print(results)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'accuracy': 0.83}


# Question Answering

In [52]:
def evaluate_pipe(pipe, dataset):
  predictions = []
  references = []
  latency = 0

  for example in dataset:
    context = example["context"]
    question = example["question"]

    t0 = time.time()
    result = pipe(question=question, context=context)
    if result["score"] < 0.1:
      print(f"\nQuestion: {question}")
      print(f"Prediction: {result['answer']}")
      print(f"Score: {result['score']}")
      print(f"Context: {context}\n")
    latency += time.time() - t0

    predictions.append({
        "id": example["id"],
        "prediction_text": result["answer"]
    })

    references.append({
        "id": example["id"],
        "answers": example["answers"]  # contains correct answers
    })

  metric = evaluate.load("squad")
  results = metric.compute(predictions=predictions, references=references)
  print(f"{results}. Avg Latency: {latency / len(dataset)}")


In [47]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

small_dataset = squad_set["validation"].shuffle(seed=7).select(range(100))

In [54]:
pipe = pipeline(
    "question-answering",
    model="bert-large-uncased-whole-word-masking-finetuned-squad",
    device=0
)

evaluate_pipe(pipe, small_dataset)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Question: What conditions must be met for a prescription for a controlled substance to be valid?
Prediction: it must be issued for a legitimate medical purpose
Score: 0.07206423580646515
Context: Of particular concern with Internet pharmacies is the ease with which people, youth in particular, can obtain controlled substances (e.g., Vicodin, generically known as hydrocodone) via the Internet without a prescription issued by a doctor/practitioner who has an established doctor-patient relationship. There are many instances where a practitioner issues a prescription, brokered by an Internet server, for a controlled substance to a "patient" s/he has never met.[citation needed] In the United States, in order for a prescription for a controlled substance to be valid, it must be issued for a legitimate medical purpose by a licensed practitioner acting in the course of legitimate doctor-patient relationship. The filling pharmacy has a corresponding responsibility to ensure that the prescripti

''exact_match': 86.0, 'f1': 93.23556005398112. Avg Latency: 0.023359265327453613

In [53]:
pipe = pipeline(
    "question-answering",
    model="deepset/minilm-uncased-squad2",
    device=0
)

evaluate_pipe(pipe, small_dataset)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: deepset/minilm-uncased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Question: The price of oil is usually a stable commodity until when?
Prediction: Until the oil shock
Score: 0.00020376457541715354
Context: This contributed to the "Oil Shock". After 1971, OPEC was slow to readjust prices to reflect this depreciation. From 1947 to 1967, the dollar price of oil had risen by less than two percent per year. Until the oil shock, the price had also remained fairly stable versus other currencies and commodities. OPEC ministers had not developed institutional mechanisms to update prices in sync with changing market conditions, so their real incomes lagged. The substantial price increases of 1973–1974 largely returned their prices and corresponding incomes to Bretton Woods levels in terms of commodities such as gold.


Question: Who ends up getting more of the teacher's resources in this scenario?
Prediction: motivated students
Score: 0.0036181090399622917
Context: Where school class sizes are typically 40 to 50 students, maintaining order in the classroom ca

'exact_match': 79.0, 'f1': 87.33945615787721. Avg Latency: 0.01183565616607666

# Observation
The "bert-large-uncased-whole-word-masking-finetuned-squad" model outperformed "minilm-uncased-squad2" but with a little over double the latency.

# Concrete Fail Case
**Question**: What conditions must be met for a prescription for a controlled substance to be valid?

**Context**: Of particular concern with Internet pharmacies is the ease with which people, youth in particular, can obtain controlled substances (e.g., Vicodin, generically known as hydrocodone) via the Internet without a prescription issued by a doctor/practitioner who has an established doctor-patient relationship. There are many instances where a practitioner issues a prescription, brokered by an Internet server, for a controlled substance to a "patient" s/he has never met.[citation needed] In the United States, in order for a prescription for a controlled substance to be valid, it must be issued for a legitimate medical purpose by a licensed practitioner acting in the course of legitimate doctor-patient relationship. The filling pharmacy has a corresponding responsibility to ensure that the prescription is valid. Often, individual state laws outline what defines a valid patient-doctor relationship.

**--First Model--**

**Prediction**: it must be issued for a legitimate medical purpose

**Score**: 0.07206423580646515

**--Second Model--**

**Prediction**: it must be issued for a legitimate medical purpose

**Score**: 0.07971037924289703
